In [1]:
# ==============================================================================
# CELL 1: Environment Setup & Dataset Verification
# ==============================================================================
!pip install -q librosa soundfile matplotlib tensorflow scikit-learn pandas

import os
import glob

DATA_DIR = "./clean_fan_dataset"
normal_files = glob.glob(f"{DATA_DIR}/*normal*.wav")
anomaly_files = glob.glob(f"{DATA_DIR}/*anomaly*.wav")

print(f"Loaded: {len(normal_files)} normal audio files")
print(f"Loaded: {len(anomaly_files)} anomaly audio files")
for f in sorted(anomaly_files):
    print("  ", os.path.basename(f))

if len(normal_files) == 0:
    raise FileNotFoundError("Place your recorded .wav files in 'clean_fan_dataset/'")


Loaded: 5 normal audio files
Loaded: 5 anomaly audio files
   anomaly_bearing_20260904_192808.wav
   anomaly_blade_striking_20260904_191438.wav
   anomaly_fast_speed_variation_20260904_192937.wav
   anomaly_obstruction_20260904_190848.wav
   anomaly_tamper_20260904_191831.wav


In [2]:
# ==============================================================================
# CELL 2: Configuration & Feature Extraction
#
# These parameters and this exact extraction logic are mirrored bit-for-bit
# in the ESP32 firmware's captureLibrosaSpectrogram(). If you change anything
# here (window size, mel count, normalization), the firmware's DSP code must
# be updated to match or the model will see out-of-distribution input.
# ==============================================================================
import numpy as np
import librosa

SAMPLE_RATE = 16000
DURATION = 1.0            # 1.0s analysis window
HOP_DURATION = 1.0         # non-overlapping windows (no leakage between train/val)
N_MELS = 32
N_FFT = 512
HOP_LENGTH = 256
TOP_DB = 80.0
TIME_FRAMES = 64           # 1.0s window -> 63 STFT frames, edge-padded to 64

# Per-file trims to cut mic handling noise / non-representative segments
# out of the raw recordings before feature extraction.
TRIM_MAP = {
    "normal_20260904_193532.wav": (5.0, None),
    "anomaly_fast_speed_variation_20260904_192937.wav": (3.0, None),
    "anomaly_tamper_20260904_191831.wav": (17.0, 35.0),
}
DISCARD_FILES = {"normal_20260904_191542.wav"}  # unstable speed throughout

def load_and_trim(path):
    y, sr = librosa.load(path, sr=SAMPLE_RATE)
    fname = os.path.basename(path)
    if fname in TRIM_MAP:
        start, end = TRIM_MAP[fname]
        start_sample = int(start * sr) if start else 0
        end_sample = int(end * sr) if end else len(y)
        y = y[start_sample:end_sample]
    return y

def extract_melspec(y_window):
    """
    One (N_MELS, TIME_FRAMES) feature, normalized to [0, 1].
    ref=np.max (not a fixed reference) is deliberate: absolute loudness
    varies session-to-session with mic placement/room acoustics and is not
    a reliable fault signal, so we normalize each window relative to its
    own peak rather than to some global level.
    """
    mel = librosa.feature.melspectrogram(
        y=y_window, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS
    )
    db = librosa.power_to_db(mel, ref=np.max, top_db=TOP_DB)
    norm = (db + TOP_DB) / TOP_DB
    norm = np.clip(norm, 0.0, 1.0)

    # Pad/trim to exactly TIME_FRAMES columns
    if norm.shape[1] < TIME_FRAMES:
        pad = TIME_FRAMES - norm.shape[1]
        norm = np.pad(norm, ((0, 0), (0, pad)), mode="edge")
    else:
        norm = norm[:, :TIME_FRAMES]
    return norm.astype(np.float32)

def windows_from_file(path):
    y = load_and_trim(path)
    win_len = int(DURATION * SAMPLE_RATE)
    hop_len = int(HOP_DURATION * SAMPLE_RATE)
    windows = []
    for start in range(0, max(1, len(y) - win_len + 1), hop_len):
        windows.append(extract_melspec(y[start:start + win_len]))
    return windows

def add_noise(spec, snr_db):
    """Additive Gaussian noise augmentation, training data only."""
    signal_power = np.mean(spec ** 2)
    noise_power = signal_power / (10 ** (snr_db / 10))
    noise = np.random.normal(0, np.sqrt(noise_power), spec.shape).astype(np.float32)
    return np.clip(spec + noise, 0.0, 1.0)

print("Feature extraction ready.")
print(f"Feature shape per window: ({N_MELS}, {TIME_FRAMES})")


Feature extraction ready.
Feature shape per window: (32, 64)


In [3]:
# ==============================================================================
# CELL 3: Build Datasets
#
# File-level train/val split (not window-level) so overlapping windows from
# the same recording never leak between train and validation.
# ==============================================================================
from sklearn.model_selection import train_test_split

usable_normal = [f for f in normal_files if os.path.basename(f) not in DISCARD_FILES]
train_files, val_files = train_test_split(usable_normal, test_size=0.2, random_state=42)

def build_normal_set(file_list, augment=False, augment_copies=2):
    specs = []
    for f in file_list:
        for w in windows_from_file(f):
            specs.append(w)
            if augment:
                for _ in range(augment_copies):
                    snr = np.random.uniform(20, 35)
                    specs.append(add_noise(w, snr))
    return np.array(specs)

X_train = build_normal_set(train_files, augment=True)
X_val = build_normal_set(val_files, augment=False)

# Anomaly windows, kept separate per file for per-fault-type evaluation later.
anomaly_windows_by_file = {
    os.path.basename(f): np.array(windows_from_file(f)) for f in anomaly_files
}

print(f"Train windows (incl. augmented): {X_train.shape}")
print(f"Val windows (normal only):       {X_val.shape}")
for name, windows in anomaly_windows_by_file.items():
    print(f"Anomaly '{name}': {windows.shape}")

X_train = X_train[..., np.newaxis]  # add channel dim -> (N, N_MELS, TIME_FRAMES, 1)
X_val = X_val[..., np.newaxis]


Train windows (incl. augmented): (312, 32, 64)
Val windows (normal only):       (53, 32, 64)
Anomaly 'anomaly_blade_striking_20260904_191438.wav': (56, 32, 64)
Anomaly 'anomaly_tamper_20260904_191831.wav': (18, 32, 64)
Anomaly 'anomaly_bearing_20260904_192808.wav': (28, 32, 64)
Anomaly 'anomaly_obstruction_20260904_190848.wav': (38, 32, 64)
Anomaly 'anomaly_fast_speed_variation_20260904_192937.wav': (19, 32, 64)


In [4]:
# ==============================================================================
# CELL 4: Autoencoder Architecture
#
# Decoder uses tf.image.resize (nearest neighbor) + Conv2D instead of
# Conv2DTranspose. The ESP32's TFLM library (TensorFlowLite_ESP32 v1.0.0,
# unmaintained since 2022) has a confirmed INT8 correctness bug in its
# TransposeConv kernel -- verified by comparing on-device output against
# Python's TFLite interpreter on identical input/weights, which disagreed
# substantially. RESIZE_NEAREST_NEIGHBOR does not have this problem.
# ==============================================================================
import tensorflow as tf
from tensorflow.keras import layers, Model

def resize_like(target_hw):
    def _resize(x):
        return tf.image.resize(x, target_hw, method="nearest")
    return layers.Lambda(_resize)

inputs = layers.Input(shape=(N_MELS, TIME_FRAMES, 1))

# Encoder
x = layers.Conv2D(8, 3, activation="relu", padding="same")(inputs)
x = layers.MaxPooling2D(2, padding="same")(x)
x = layers.Conv2D(4, 3, activation="relu", padding="same")(x)
x = layers.MaxPooling2D(2, padding="same")(x)
bottleneck = layers.Conv2D(2, 3, activation="relu", padding="same")(x)  # (8, 16, 2)

# Decoder
x = resize_like((16, 32))(bottleneck)
x = layers.Conv2D(4, 3, activation="relu", padding="same")(x)
x = resize_like((32, 64))(x)
x = layers.Conv2D(8, 3, activation="relu", padding="same")(x)
outputs = layers.Conv2D(1, 3, activation="sigmoid", padding="same")(x)

autoencoder = Model(inputs, outputs, name="fan_anomaly_autoencoder")
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()


Model: "fan_anomaly_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 64, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 32, 64, 8)      │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 32, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 32, 4)      │           292 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 16, 4)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 16, 2)       │            74 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 16, 32, 2)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 32, 4)      │            76 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, 32, 64, 4)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 64, 8)      │           296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 64, 1)      │            73 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 891 (3.48 KB)

 Trainable params: 891 (3.48 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# ==============================================================================
# CELL 5: Train
# ==============================================================================
history = autoencoder.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=100,
    batch_size=16,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    ],
    verbose=2,
)


Epoch 1/100
20/20 - 6s - 317ms/step - loss: 0.0536 - val_loss: 0.0398
Epoch 2/100
20/20 - 1s - 68ms/step - loss: 0.0469 - val_loss: 0.0283
Epoch 3/100
20/20 - 1s - 73ms/step - loss: 0.0276 - val_loss: 0.0165
Epoch 4/100
20/20 - 1s - 69ms/step - loss: 0.0190 - val_loss: 0.0117
Epoch 5/100
20/20 - 2s - 122ms/step - loss: 0.0136 - val_loss: 0.0089
Epoch 6/100
20/20 - 2s - 97ms/step - loss: 0.0102 - val_loss: 0.0065
Epoch 7/100
20/20 - 1s - 68ms/step - loss: 0.0089 - val_loss: 0.0064
Epoch 8/100
20/20 - 1s - 70ms/step - loss: 0.0075 - val_loss: 0.0058
Epoch 9/100
20/20 - 1s - 67ms/step - loss: 0.0067 - val_loss: 0.0055
Epoch 10/100
20/20 - 2s - 99ms/step - loss: 0.0056 - val_loss: 0.0039
Epoch 11/100
20/20 - 1s - 31ms/step - loss: 0.0050 - val_loss: 0.0039
Epoch 12/100
20/20 - 1s - 31ms/step - loss: 0.0048 - val_loss: 0.0038
Epoch 13/100
20/20 - 1s - 31ms/step - loss: 0.0046 - val_loss: 0.0037
Epoch 14/100
20/20 - 1s - 31ms/step - loss: 0.0044 - val_loss: 0.0036
Epoch 15/100
20/20 - 1s - 3

In [6]:
# ==============================================================================
# CELL 6: Evaluate
# ==============================================================================
from sklearn.metrics import roc_auc_score

def reconstruction_mse(model, X):
    recon = model.predict(X, verbose=0)
    return np.mean((X - recon) ** 2, axis=(1, 2, 3))

normal_mse = reconstruction_mse(autoencoder, X_val)
print(f"Normal (val) MSE: mean={normal_mse.mean():.5f}, std={normal_mse.std():.5f}")

all_anomaly_mse = []
print("\nPer-fault-type reconstruction error:")
for name, windows in anomaly_windows_by_file.items():
    X_fault = windows[..., np.newaxis]
    fault_mse = reconstruction_mse(autoencoder, X_fault)
    all_anomaly_mse.append(fault_mse)
    labels = np.concatenate([np.zeros_like(normal_mse), np.ones_like(fault_mse)])
    scores = np.concatenate([normal_mse, fault_mse])
    auc = roc_auc_score(labels, scores)
    print(f"  {name:55s}  mean_mse={fault_mse.mean():.5f}  AUC={auc:.3f}")

all_anomaly_mse = np.concatenate(all_anomaly_mse)
pooled_labels = np.concatenate([np.zeros_like(normal_mse), np.ones_like(all_anomaly_mse)])
pooled_scores = np.concatenate([normal_mse, all_anomaly_mse])
print(f"\nPooled AUC (all fault types vs normal): {roc_auc_score(pooled_labels, pooled_scores):.4f}")


Normal (val) MSE: mean=0.00203, std=0.00041

Per-fault-type reconstruction error:
  anomaly_blade_striking_20260904_191438.wav               mean_mse=0.00336  AUC=0.869
  anomaly_tamper_20260904_191831.wav                       mean_mse=0.01013  AUC=0.995
  anomaly_bearing_20260904_192808.wav                      mean_mse=0.00302  AUC=0.813
  anomaly_obstruction_20260904_190848.wav                  mean_mse=0.00273  AUC=0.613
  anomaly_fast_speed_variation_20260904_192937.wav         mean_mse=0.00161  AUC=0.142

Pooled AUC (all fault types vs normal): 0.7252


In [7]:
# ==============================================================================
# CELL 7: INT8 Quantization & Export
#
# The final Conv2D(1, activation="sigmoid") layer is split into a linear
# Conv2D (same trained weights) here, with sigmoid applied manually in the
# firmware's C++ code instead. This works around a confirmed bug in this
# TFLM library's on-device Logistic (sigmoid) kernel -- verified by reading
# the pre-sigmoid tensor directly from both the device and Python's TFLite
# interpreter on identical input: the Conv2D math matched to 4 decimal
# places, but the device's actual sigmoid OUTPUT did not, isolating the bug
# to that one kernel. Every other op in the graph (Conv2D, MaxPool2D,
# ResizeNearestNeighbor) was individually verified to match Python exactly.
# ==============================================================================
final_layer = autoencoder.layers[-1]  # Conv2D(1, activation="sigmoid")
weights, biases = final_layer.get_weights()

pre_final_output = autoencoder.layers[-2].output
presigmoid_layer = layers.Conv2D(
    filters=final_layer.filters,
    kernel_size=final_layer.kernel_size,
    padding=final_layer.padding,
    activation=None,
    name="presigmoid_output",
)
presigmoid_output = presigmoid_layer(pre_final_output)
presigmoid_model = Model(inputs=autoencoder.input, outputs=presigmoid_output)
presigmoid_layer.set_weights([weights, biases])

def representative_dataset():
    for i in range(min(300, len(X_train))):
        yield [X_train[i:i + 1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(presigmoid_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_model = converter.convert()

with open("presigmoid_model.tflite", "wb") as f:
    f.write(tflite_model)
int8_size = len(tflite_model)
print(f"Wrote presigmoid_model.tflite ({int8_size} bytes)")

# --- Size comparison: same graph, float32 vs INT8 ---
float_converter = tf.lite.TFLiteConverter.from_keras_model(presigmoid_model)
float_tflite_model = float_converter.convert()
float_size = len(float_tflite_model)

size_reduction_pct = (1 - int8_size / float_size) * 100
print(f"\nFloat32 TFLite size: {float_size:,} bytes")
print(f"INT8    TFLite size: {int8_size:,} bytes")
print(f"Size reduction from INT8 quantization: {size_reduction_pct:.1f}%")

# Sanity check: confirm the exported INT8 model still separates normal from
# anomaly the same way the float model did.
interp = tf.lite.Interpreter(model_path="presigmoid_model.tflite")
interp.allocate_tensors()
in_details = interp.get_input_details()[0]
out_details = interp.get_output_details()[0]
in_scale, in_zp = in_details["quantization"]
out_scale, out_zp = out_details["quantization"]

def int8_reconstruction_mse(X):
    mses = []
    for x in X:
        x_q = np.round(x[np.newaxis] / in_scale + in_zp).astype(np.int8)
        interp.set_tensor(in_details["index"], x_q)
        interp.invoke()
        raw = interp.get_tensor(out_details["index"])[0]
        presigmoid = (raw.astype(np.float32) - out_zp) * out_scale
        recon = 1.0 / (1.0 + np.exp(-presigmoid))
        mses.append(np.mean((x - recon) ** 2))
    return np.array(mses)

int8_normal_mse = int8_reconstruction_mse(X_val)
print(f"INT8 normal MSE: mean={int8_normal_mse.mean():.5f} (float was {normal_mse.mean():.5f})")



Saved artifact at '/tmp/tmpgp0qrf6t'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 64, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 32, 64, 1), dtype=tf.float32, name=None)
Captures:
  135955718143824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135955718143632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135955718143056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135955423872208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135955423871632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135955423871440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135955423872976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135955423871056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135955423872592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135955423873552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13595542

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Wrote presigmoid_model.tflite (6752 bytes)
Saved artifact at '/tmp/tmpl5apjqlp'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 64, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 32, 64, 1), dtype=tf.float32, name=None)
Captures:
  135955718143824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135955718143632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135955718143056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135955423872208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135955423871632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135955423871440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135955423872976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135955423871056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135955423872592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135955423873552: TensorSpec(shape=(

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [8]:
# ==============================================================================
# CELL 8: Generate C Header for Firmware
# ==============================================================================
def tflite_to_c_header(tflite_path, header_path, array_name):
    with open(tflite_path, "rb") as f:
        data = f.read()
    with open(header_path, "w") as f:
        f.write(f"const unsigned char {array_name}[] = {{\n")
        for i in range(0, len(data), 12):
            chunk = data[i:i + 12]
            f.write("  " + ", ".join(f"0x{b:02x}" for b in chunk) + ",\n")
        f.write("};\n")
        f.write(f"const unsigned int {array_name}_len = {len(data)};\n")
    print(f"Wrote {header_path} ({len(data)} bytes -> {array_name})")

tflite_to_c_header("presigmoid_model.tflite", "presigmoid_model_data.h", "presigmoid_model_tflite")


Wrote presigmoid_model_data.h (6752 bytes -> presigmoid_model_tflite)


In [9]:
# ==============================================================================
# CELL 9: Precision / Recall / F1 -- Float vs. INT8, at deployed-style thresholds
#
# AUC alone doesn't tell you how the system behaves at the ONE threshold
# it's actually deployed with, and says nothing about whether quantization
# cost any real detection performance. This computes both, side by side.
#
# Thresholds are derived from each model's OWN normal-only distribution
# (mean +/- 3*std), matching the two-sided calibration approach used in the
# firmware -- NOT the firmware's literal threshold constants, since Python's
# MSE and the on-device MSE are not on the same absolute scale (mic noise,
# room acoustics, and INT8 rounding shift it) -- only the relative,
# distribution-based calibration methodology transfers.
# ==============================================================================
from sklearn.metrics import precision_score, recall_score, f1_score

def two_sided_thresholds(normal_mse, k=3.0):
    mean, std = normal_mse.mean(), normal_mse.std()
    return mean - k * std, mean + k * std

def evaluate_at_thresholds(normal_mse, anomaly_mse_by_file, low, high):
    """Pools all normal + all anomaly windows into one binary detection task."""
    y_true, y_pred = [], []
    for mse in normal_mse:
        y_true.append(0)
        y_pred.append(1 if (mse < low or mse > high) else 0)
    for fault_mse in anomaly_mse_by_file.values():
        for mse in fault_mse:
            y_true.append(1)
            y_pred.append(1 if (mse < low or mse > high) else 0)
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "false_positive_rate": sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1) / max(1, sum(1 for t in y_true if t == 0)),
    }

# --- Float model ---
float_anomaly_mse = {
    name: reconstruction_mse(autoencoder, windows[..., np.newaxis])
    for name, windows in anomaly_windows_by_file.items()
}
float_low, float_high = two_sided_thresholds(normal_mse)
float_results = evaluate_at_thresholds(normal_mse, float_anomaly_mse, float_low, float_high)

# --- INT8 model ---
int8_anomaly_mse = {
    name: int8_reconstruction_mse(windows[..., np.newaxis])
    for name, windows in anomaly_windows_by_file.items()
}
int8_low, int8_high = two_sided_thresholds(int8_normal_mse)
int8_results = evaluate_at_thresholds(int8_normal_mse, int8_anomaly_mse, int8_low, int8_high)

print(f"{'Metric':<22} {'Float':>10} {'INT8':>10}")
print("-" * 44)
for metric in ["precision", "recall", "f1", "false_positive_rate"]:
    print(f"{metric:<22} {float_results[metric]:>10.3f} {int8_results[metric]:>10.3f}")

print(f"\nFloat thresholds:  [{float_low:.5f}, {float_high:.5f}]  (from mean={normal_mse.mean():.5f}, std={normal_mse.std():.5f})")
print(f"INT8  thresholds:  [{int8_low:.5f}, {int8_high:.5f}]  (from mean={int8_normal_mse.mean():.5f}, std={int8_normal_mse.std():.5f})")
print(f"\nQuantization error on normal baseline: {abs(int8_normal_mse.mean() - normal_mse.mean()) / normal_mse.mean() * 100:.1f}% relative shift in mean MSE")


Metric                      Float       INT8
--------------------------------------------
precision                   0.984      0.984
recall                      0.384      0.396
f1                          0.552      0.565
false_positive_rate         0.019      0.019

Float thresholds:  [0.00079, 0.00328]  (from mean=0.00203, std=0.00041)
INT8  thresholds:  [0.00079, 0.00327]  (from mean=0.00203, std=0.00041)

Quantization error on normal baseline: 0.2% relative shift in mean MSE
